# Two-Step Task

## 1. Load the data

In [10]:
import pandas as pd
import plotly.graph_objects as go

data = pd.read_csv("./reduced_two_step_mixture_data.csv")
data.head()

,trial,controller,choice_stage1,state_stage2,transition,reward,reward_prob
0,1,model-free,0,0,common,0,0.634224
1,2,model-based,0,0,common,1,0.658748
2,3,model-based,0,0,common,1,0.636525
3,4,model-free,0,0,common,0,0.618589
4,5,model-free,0,0,common,0,0.625533


In [11]:
model_free_data = data[data["controller"] == "model-free"]
model_based_data = data[data["controller"] == "model-based"]
print(len(model_free_data), len(model_based_data))

380 620


## 2. Model Free Approach

In [12]:
class Model:
    def __init__(self, alpha=0.5):
        self.alpha = alpha
        self.v_table = {}

    def get_value(self, state):
        if state not in self.v_table:
            self.v_table[state] = 0
        return self.v_table[state]

    def update(self, state, reward):
        current_value = self.get_value(state)
        delta_v = self.alpha * (reward - current_value)
        self.v_table[state] = current_value + delta_v

In [13]:
p_common_reward = []
p_common_unreward = []
p_rare_reward = []
p_rare_unreward = []

v0 = []
v1 = []

model_free_model = Model()
for i, row in model_free_data.iterrows():
    state = row["choice_stage1"]
    reward = row["reward"]
    transition = row["transition"]
    model_free_model.update(state, reward)

    state0_value = model_free_model.get_value(0)
    state1_value = model_free_model.get_value(1)
    v0.append(state0_value)
    v1.append(state1_value)
    if state0_value == 0 and state1_value == 0:
        p = 0
    else:
        p = state0_value / (state0_value + state1_value)
    if transition == "common" and reward == 1:
        p_common_reward.append(p if state == 0 else 1 - p)
    elif transition == "common" and reward == 0:
        p_common_unreward.append(p if state == 0 else 1 - p)
    elif transition == "rare" and reward == 1:
        p_rare_reward.append(p if state == 0 else 1 - p)
    elif transition == "rare" and reward == 0:
        p_rare_unreward.append(p if state == 0 else 1 - p)

p_common_reward = sum(p_common_reward) / len(p_common_reward)
p_common_unreward = sum(p_common_unreward) / len(p_common_unreward)
p_rare_reward = sum(p_rare_reward) / len(p_rare_reward)
p_rare_unreward = sum(p_rare_unreward) / len(p_rare_unreward)

In [14]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_common_reward, p_common_unreward], name="common"))
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_rare_reward, p_rare_unreward], name="rare"))
fig.update_layout(barmode="group", title="Model-Free Approach", width=1000, height=600)
fig.show()

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[i for i in range(1, len(model_free_data) + 1)], y=v0, name="state 0"))
fig.add_trace(go.Scatter(x=[i for i in range(1, len(model_free_data) + 1)], y=v1, name="state 1"))
fig.update_layout(title="Model-Free Approach", width=1000, height=600)
fig.show()

## 3. Model-Based Approach

In [16]:
p_common_reward = []
p_common_unreward = []
p_rare_reward = []
p_rare_unreward = []

v0 = []
v1 = []

model_based_model = Model()
for i, row in model_based_data.iterrows():
    state = row["choice_stage1"]
    next_state = row["state_stage2"]
    reward = row["reward"]
    transition = row["transition"]
    model_based_model.update(next_state, reward)

    state0_value = model_based_model.get_value(0)
    state1_value = model_based_model.get_value(1)
    v0.append(state0_value)
    v1.append(state1_value)
    if state0_value == 0 and state1_value == 0:
        p = 0
    else:
        p = state0_value / (state0_value + state1_value)
    if transition == "common" and reward == 1:
        p_common_reward.append(p if state == 0 else 1 - p)
    elif transition == "common" and reward == 0:
        p_common_unreward.append(p if state == 0 else 1 - p)
    elif transition == "rare" and reward == 1:
        p_rare_reward.append(p if state == 0 else 1 - p)
    elif transition == "rare" and reward == 0:
        p_rare_unreward.append(p if state == 0 else 1 - p)

p_common_reward = sum(p_common_reward) / len(p_common_reward)
p_common_unreward = sum(p_common_unreward) / len(p_common_unreward)
p_rare_reward = sum(p_rare_reward) / len(p_rare_reward)
p_rare_unreward = sum(p_rare_unreward) / len(p_rare_unreward)

In [17]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_common_reward, p_common_unreward], name="common"))
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_rare_reward, p_rare_unreward], name="rare"))
fig.update_layout(barmode="group", title="Model-Based Approach", width=1000, height=600)
fig.show()

In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[i for i in range(1, len(model_based_data) + 1)], y=v0, name="state 0"))
fig.add_trace(go.Scatter(x=[i for i in range(1, len(model_based_data) + 1)], y=v1, name="state 1"))
fig.update_layout(title="Model-Based Approach", width=1000, height=600)
fig.show()